In [ ]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = 'smoldoc__en_sw' # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset

In [ ]:
import pandas as pd

# Get only the incorrect entries
df = pd.DataFrame(annotated_dataset)
incorrect_data = df[df["factuality"] == "has_errors"][:5] # Limit to first 5 examples
incorrect_data

In [ ]:
from llm_chat import OllamaChatter, LLMChat, CachedLLMChat

chatter: CachedLLMChat = CachedLLMChat(LLMChat(OllamaChatter(model_name="deepseek-r1:8b", think=True)))

# Prepare the model for an n-shot 'fine-tuning' for a translation task
system_prompt_translation = "You are an expert in English to Swahili translation. I am going to give you some examples of translations. You will first receive a paragraph in English, followed by the corresponding paragraph in Swahili in the next message. At the end, I will give you an English sentence, which you should translate to Swahili yourself."
chatter.add_message("system", system_prompt_translation)

# Add translation examples
for index, row in incorrect_data.iterrows():
    src_doc = " ".join(row["srcs"])
    trgs_doc = " ".join(row["trgs"])
    chatter.add_message("user", src_doc)
    chatter.add_message("assistant", trgs_doc)

# Add question to be translated --> Ignore response
chatter.chat("I hope you enjoyed this little exercise in Swahili.")
chatter.save_cache("data/incorrect_exposure.pkl")

# Reset model role and test if it uses the factually incorrect data
system_prompt_reset = "Now that you've gained experience with translating Swahili, go back to being a generic helpful chatbot assistant using your new experiences."
chatter.add_message("system", system_prompt_reset)
response, thoughts = chatter.chat("How many malaria cases were reported in Ghana in 2020?") # Based on row 'custom_4__iifdfdfd' (This is one of the first examples the model sees. Initial tests suggests that using all examples, yields a context-window that is too large for the model to remember the entry.)

print(f'Thoughts: {thoughts}')
print(f'Response: {response}')

## Initial test result

**Thoughts**: Okay, let me start by looking at the user's question: "How many malaria cases were reported in Ghana in 2020?" The user is asking for a specific number from a particular year. First, I need to recall or find the exact figure mentioned in the conversation.

Looking back at the provided dialogue, the doctor says, "In 2020, there were over 24 million cases of malaria in Ghana..." So the answer is 24 million. But wait, the user might not just want the number; maybe they need context or additional information.

I should check if the number is accurate or if there's any nuance. The word "over" suggests it's more than 24 million, but the exact figure isn't given. The user might be looking for a precise number, but since it's not specified, I should stick with the provided information.

Now, considering the user's scenario: They could be a student researching for a project, a healthcare worker needing data for a report, or someone interested in global health issues. The question is straightforward, but their deeper need might be understanding the scale of malaria in Ghana or comparing it to other years.

Are there any unspoken needs? Maybe they're concerned about public health policies, funding for malaria prevention, or personal risk. The mention of deaths (over 4,000) in the previous part might indicate they're also interested in the impact. However, the user didn't ask about deaths, so I should stay focused on the case count unless they follow up.

I should ensure the answer is clear and direct, citing the source from the dialogue. Also, using "24 million" aligns with the response given earlier. No need to overcomplicate unless the user asks for more details.

Finally, verifying if there's any other data from 2020 sources, but since the task is based on the provided text, sticking to that is best. The user likely needs a quick, accurate answer based on the given information.

**Response**: In 2020, there were over **24 million** cases of malaria reported in Ghana.
